<a href="https://colab.research.google.com/github/gunnsmart/skills/blob/main/AI_Image_Upscaler_for_Adobe_Stock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#  AI Image Upscaler for Adobe Stock
> **Real-ESRGAN x4 Plus** | ฟรี 100% บน Google Colab ไม่กินสเปกเครื่อง

| Cell | หน้าที่ |
|---|---|
| **Cell 1** | ติดตั้งระบบขยายภาพ Real-ESRGAN |
| **Cell 2** | เชื่อมต่อ Google Drive + ตั้งค่าโฟลเดอร์ |
| **Cell 3** | สั่งรัน Upscale ภาพทั้งหมดแบบอัตโนมัติ |

###  โครงสร้างโฟลเดอร์ใน Google Drive

My Drive/AdobeStock_Upscaler/
├── input/        ← เอารูปต้นฉบับมาวางที่นี่
└── output/       ← รูปที่ขยายเสร็จแล้วจะมาเซฟที่นี่อัตโนมัติ

>  **ข้อสำคัญก่อนรัน:** ไปที่เมนู เมนู Runtime → Change runtime type → เลือกเป็น **T4 GPU** ด้วยนะครับ

In [ ]:
# ============================================================
# CELL 1 — ติดตั้ง Real-ESRGAN
# ============================================================
import subprocess, sys, os, site, pathlib

print('📦 ติดตั้งแพ็คเกจ...')
pkgs = ['basicsr', 'facexlib', 'gfpgan', 'realesrgan', 'Pillow', 'opencv-python', 'pandas']
subprocess.run([sys.executable, '-m', 'pip', 'install'] + pkgs + ['-q'])

# patch basicsr torchvision เพื่อป้องกันเออเร่อ
for sp in site.getsitepackages():
    deg = pathlib.Path(sp) / 'basicsr/data/degradations.py'
    if deg.exists():
        txt = deg.read_text()
        if 'functional_tensor' in txt:
            deg.write_text(txt.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            ))
            print('  🔧 patched basicsr')
        break

if not os.path.exists('/content/Real-ESRGAN'):
    subprocess.run(['git', 'clone', 'https://github.com/xinntao/Real-ESRGAN.git', '-q'])

os.chdir('/content/Real-ESRGAN')
subprocess.run([sys.executable, 'setup.py', 'develop', '-q'], capture_output=True)
os.makedirs('results', exist_ok=True)
os.makedirs('inputs',  exist_ok=True)
print('✅ ระบบติดตั้งพร้อมใช้งาน! รัน Cell 2 ต่อได้เลย')

In [ ]:
# ============================================================
# CELL 2 — เชื่อม Google Drive + ตั้งค่า
# ============================================================
from google.colab import drive
from PIL import Image
import os, glob

drive.mount('/content/drive')

# ✅ กำหนดสิทธิ์และ Paths ในการอ่าน/เขียนไฟล์
GDRIVE_BASE   = '/content/drive/MyDrive/AdobeStock_Upscaler'
INPUT_FOLDER  = GDRIVE_BASE + '/input'
OUTPUT_FOLDER = GDRIVE_BASE + '/output'
ESRGAN_DIR    = '/content/Real-ESRGAN'
RESULTS_DIR   = ESRGAN_DIR + '/results'
INPUTS_DIR    = ESRGAN_DIR + '/inputs'

for d in [INPUT_FOLDER, OUTPUT_FOLDER, RESULTS_DIR, INPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ✅ ตั้งค่าการอัปสเกล (ปรับแต่งได้ที่นี่)
ESRGAN_MODEL  = 'RealESRGAN_x4plus'
SCALE         = 4
JPEG_QUALITY  = 95
TARGET_DPI    = 300
SUFFIX        = 'stockai'   # ← ใส่ชื่อหรือนามปากกาที่จะให้ลงท้ายชื่อไฟล์ภาพ

# ✅ ตรวจเช็กไฟล์ภาพในโฟลเดอร์ input
IMG_EXTS = ('*.jpg','*.jpeg','*.png','*.webp','*.bmp','*.JPG','*.PNG','*.JPEG')
imgs = []
for ext in IMG_EXTS:
    imgs.extend(glob.glob(INPUT_FOLDER + '/' + ext))
imgs = sorted(set(imgs))

print('✅ เชื่อมต่อ Google Drive สำเร็จ')
print('📂 โฟลเดอร์ต้นฉบับ (Input)  :', INPUT_FOLDER)
print('📂 โฟลเดอร์ปลายทาง (Output) :', OUTPUT_FOLDER)
print(f'\n📸 ค้นพบไฟล์ภาพทั้งหมด {len(imgs)} ภาพในระบบ')
for p in imgs:
    with Image.open(p) as im: w, h = im.size
    name = os.path.basename(p)
    stem = os.path.splitext(name)[0]
    out_name = f'{stem}_{SUFFIX}.jpg'
    print(f'  {name}  →  {out_name}  ({w}x{h} → {w*SCALE}x{h*SCALE})')
print('\n👉 ตรวจสอบความถูกต้องแล้วกดรัน Cell 3 ได้เลย!')

In [ ]:
# ============================================================
# CELL 3 — สั่งรัน Upscale ภาพอัตโนมัติ
# ============================================================
import subprocess, shutil, time, os
from PIL import Image

os.chdir(ESRGAN_DIR)
skip_count = 0

print(f'📥 เริ่มกระบวนการ Upscale ภาพจำนวน {len(imgs)} ภาพ  →  เซฟเข้าโฟลเดอร์ {OUTPUT_FOLDER}')
print(f'📝 รูปแบบชื่อไฟล์ใหม่: {{adobeai}}_{SUFFIX}.jpg\n')

for idx, src_path in enumerate(imgs):
    src_name  = os.path.basename(src_path)
    src_stem  = os.path.splitext(src_name)[0]
    out_name  = f'{src_stem}_{SUFFIX}.jpg'
    out_path  = os.path.join(OUTPUT_FOLDER, out_name)

    # ข้ามรูปภาพถ้าตรวจสอบพบว่าเคยแปลงไฟล์เสร็จไปแล้ว
    if os.path.exists(out_path):
        print(f'[{idx+1}/{len(imgs)}] ⏩ ข้ามภาพนี้ (ตรวจพบไฟล์เดิมอยู่แล้ว): {out_name}')
        skip_count += 1
        continue

    with Image.open(src_path) as im:
        w, h = im.size

    eta = '' if idx == 0 else f'ETA: ~{int((time.time()-t_start)/(idx)*(len(imgs)-idx)//60)} นาที'
    if idx == 0: t_start = time.time()
    print(f'[{idx+1}/{len(imgs)}] 🖼️  กำลังประมวลผล: {src_name}  {eta}')
    print(f'  📐 ความละเอียดเดิม {w}x{h} px → ปรับขึ้นเป็น {w*SCALE}x{h*SCALE} px')

    # ย้ายไฟล์ภาพชั่วคราวเข้าไปยังหน่วยประมวลผล
    tmp_input = os.path.join(INPUTS_DIR, src_name)
    shutil.copy2(src_path, tmp_input)

    # เคลียร์ขยะในโฟลเดอร์ผลลัพธ์เก่าออกก่อน
    for f in os.listdir(RESULTS_DIR):
        os.remove(os.path.join(RESULTS_DIR, f))

    # ป้องกันการเกิดอาการหน่วยความจำ GPU เต็ม (Out of Memory) สำหรับภาพขนาดใหญ่
    tile = 512 if w * h > 3000000 else 0

    t0 = time.time()
    for tile_size in ([tile] if tile else [0, 256, 512]):
        cmd = [
            'python', 'inference_realesrgan.py',
            '-n', ESRGAN_MODEL,
            '-i', tmp_input,
            '-o', RESULTS_DIR,
            '-s', str(SCALE),
        ]
        if tile_size > 0:
            cmd += ['--tile', str(tile_size)]

        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            break
        elif 'out of memory' in result.stderr.lower():
            print(f'  ⚠️ การ์ดจอเต็ม กำลังปรับขนาด Tile อัตโนมัติ ({tile_size+256})...')
            tile_size += 256
        else:
            print(f'❌ ค้นพบข้อผิดพลาด: {result.stderr[-200:]}')
            break

    # นำไฟล์ผลลัพธ์ที่ได้ไปเซฟลง Google Drive
    result_files = os.listdir(RESULTS_DIR)
    if not result_files:
        print(f'❌ ไม่พบไฟล์ผลลัพธ์ที่แปลงเสร็จสิ้นในระบบ')
        continue

    result_file = os.path.join(RESULTS_DIR, result_files[0])

    # แปลงไฟล์ภาพและบีบอัดเป็นค่ามาตรฐาน JPEG คุณภาพสูง ฝัง DPI 300 สำหรับงานพิมพ์ส่ง Adobe Stock
    with Image.open(result_file) as im:
        im_rgb = im.convert('RGB')
        im_rgb.save(out_path, 'JPEG', quality=JPEG_QUALITY,
                    dpi=(TARGET_DPI, TARGET_DPI), optimize=True)

    elapsed = time.time() - t0
    size_mb = os.path.getsize(out_path) / 1000000
    print(f'✅ แปลงไฟล์สำเร็จ: {out_name} ({size_mb:.1f}MB) ใช้เวลา {elapsed:.1f} วินาที')

    # ล้างไฟล์ภาพชั่วคราวทิ้ง
    os.remove(tmp_input)

print(f'\n🎉 🎉 [เสร็จสิ้นภารกิจ!] อัปสเกลครบทั้งหมดเรียบร้อยแล้ว (ข้ามไฟล์เก่าไป {skip_count} ภาพ)')